## Google Colab Setup

This notebook can run either locally from the repo or on Google Colab.

Colab workflow:
- Mount Google Drive and save all trained-model outputs there immediately.
- Download only the exact code, data, and pre-computed structural-feature files needed into `/content/XAllergen2.0`.
- Download a fresh ESM-2 snapshot from Hugging Face.
- Avoid `git clone`, which can be less reliable in hosted notebook runtimes.

Required GitHub-hosted artifacts:
- `models/baseline_frozen_esm2.pt`
- `data/ss3/deepalgpro_train_ss3_structured.json.gz` / `deepalgpro_test_ss3_structured.json.gz`

Section C reads `results/ss3_regularization/sweep_summary.csv` produced by notebook 13  
and `results/rsa_regularization/sweep_summary.csv` produced by notebook 12.  
On Colab these are read from Google Drive; run notebooks 12 and 13 first.

In [2]:
import os
import sys
import urllib.request
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
os.environ['XALLERGEN_RUN_TARGET'] = 'colab' if IS_COLAB else 'local'

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/XAllergen2.0')
else:
    DRIVE_ROOT = None

if IS_COLAB:
    RUNTIME_ROOT = Path('/content/XAllergen2.0')
else:
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        if (_candidate / 'src' / 'xallergen').exists():
            RUNTIME_ROOT = _candidate
            break
    else:
        raise FileNotFoundError('Could not locate repo root with src/xallergen')

SRC_DIR              = RUNTIME_ROOT / 'src'
PACKAGE_DIR          = SRC_DIR / 'xallergen'
DATA_DIR             = RUNTIME_ROOT / 'data'
SS3_DIR              = DATA_DIR / 'ss3'
RUNTIME_MODELS_DIR   = RUNTIME_ROOT / 'models'
RUNTIME_RESULTS_DIR  = RUNTIME_ROOT / 'results'
HF_DIR               = RUNTIME_ROOT / 'hf_models' / 'facebook_esm2_t6_8M_UR50D'

if IS_COLAB:
    MODELS_DIR  = DRIVE_ROOT / 'models'
    RESULTS_DIR = DRIVE_ROOT / 'results'
else:
    MODELS_DIR  = RUNTIME_MODELS_DIR
    RESULTS_DIR = RUNTIME_RESULTS_DIR

for path in [
    SRC_DIR, PACKAGE_DIR, DATA_DIR, SS3_DIR,
    RUNTIME_MODELS_DIR, RUNTIME_RESULTS_DIR, MODELS_DIR, RESULTS_DIR, HF_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from huggingface_hub import snapshot_download

    RAW = 'https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main'

    package_files = [
        '__init__.py',
        'baseline_notebook_utils.py',
        'baseline_sasa_experiment.py',
        'mtl_epitope_notebook_utils.py',
        'rsa_preprocessing.py',
    ]
    data_files = [
        'deepalgpro_train_cleaned.csv',
        'deepalgpro_test_cleaned.csv',
        'positives_splitB.csv',
    ]
    ss3_files = [
        'deepalgpro_train_ss3_structured.json.gz',
        'deepalgpro_test_ss3_structured.json.gz',
    ]

    download_jobs = []
    download_jobs.extend((f'{RAW}/src/xallergen/{name}', PACKAGE_DIR / name) for name in package_files)
    download_jobs.extend((f'{RAW}/data/{name}', DATA_DIR / name) for name in data_files)
    download_jobs.extend((f'{RAW}/data/ss3/{name}', SS3_DIR / name) for name in ss3_files)
    download_jobs.append((f'{RAW}/models/baseline_frozen_esm2.pt', RUNTIME_MODELS_DIR / 'baseline_frozen_esm2.pt'))

    failed_urls = []
    for url, dst in download_jobs:
        try:
            urllib.request.urlretrieve(url, dst)
        except Exception as exc:
            failed_urls.append((url, str(exc)))

    if failed_urls:
        details = '\n'.join(f'  - {url} -> {err}' for url, err in failed_urls)
        raise RuntimeError('Failed to download required GitHub artifacts:\n' + details)

    fresh_model_path = snapshot_download(
        repo_id='facebook/esm2_t6_8M_UR50D',
        local_dir=HF_DIR,
        local_dir_use_symlinks=False,
        force_download=True,
        resume_download=False,
    )
    os.environ['XALLERGEN_HF_MODEL_DIR'] = str(fresh_model_path)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'RUN_TARGET: {os.environ["XALLERGEN_RUN_TARGET"]}')
print(f'Runtime root: {RUNTIME_ROOT}')
print(f'SS3 dir: {SS3_DIR}')
print(f'Output results dir: {RESULTS_DIR}')

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

RUN_TARGET: colab
Runtime root: /content/XAllergen2.0
SS3 dir: /content/XAllergen2.0/data/ss3
Output results dir: /content/drive/MyDrive/XAllergen2.0/results


# SS3-Structured Regularization — Extended High-λ Sweep

Notebook 13 found that SS3-structured regularization (λ ∈ {0.1, 0.5, 1, 5}) shows monotonically
improving validation MCC and residue AUROC, with the best result at the top of the sweep (λ = 5).
This suggests the optimum may lie at higher λ values.

This notebook extends the sweep to λ ∈ {7, 10, 15, 20} using the same setup.
Results are saved to `results/ss3_high_lambda/` and compared against
the baseline and the earlier sweep in Section B.

In [3]:
import importlib
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'pyproject.toml').exists() or (candidate / 'src' / 'xallergen').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root containing pyproject.toml or src/xallergen')


if 'DATA_DIR' not in globals():
    RUNTIME_ROOT = _find_repo_root(Path.cwd())
    SRC_DIR      = RUNTIME_ROOT / 'src'
    DATA_DIR     = RUNTIME_ROOT / 'data'
    SS3_DIR      = DATA_DIR / 'ss3'
    RUNTIME_MODELS_DIR = RUNTIME_ROOT / 'models'
    RESULTS_DIR  = RUNTIME_ROOT / 'results'
    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))

missing_modules = []
for module_name in ('numpy', 'pandas', 'torch', 'IPython', 'sklearn'):
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_modules.append(module_name)

if missing_modules:
    missing = ', '.join(missing_modules)
    raise ModuleNotFoundError(
        'Notebook dependencies are missing from the active kernel: '
        f'{missing}. Run `make setup` and select `.venv/bin/python` or the '
        '`Python (xallergen2)` kernel before rerunning this cell.'
    )

import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split

from xallergen.baseline_notebook_utils import RANDOM_STATE, seed_everything
from xallergen.baseline_sasa_experiment import (
    RSARegularizationConfig,
    inspect_rsa_inputs,
    load_rsa_lookup_dicts,
    run_lambda_rsa_sweep,
)

TRAIN_CSV                = DATA_DIR / 'deepalgpro_train_cleaned.csv'
TEST_CSV                 = DATA_DIR / 'deepalgpro_test_cleaned.csv'
POSITIVES_SPLITB_CSV     = DATA_DIR / 'positives_splitB.csv'
BASELINE_CHECKPOINT_PATH = RUNTIME_MODELS_DIR / 'baseline_frozen_esm2.pt'
TRAIN_SS3_PATH           = SS3_DIR / 'deepalgpro_train_ss3_structured.json.gz'
TEST_SS3_PATH            = SS3_DIR / 'deepalgpro_test_ss3_structured.json.gz'

SS3_HIGH_RESULTS_DIR = RESULTS_DIR / 'ss3_high_lambda'
SS3_HIGH_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SS3_HIGH_SWEEP_CSV = SS3_HIGH_RESULTS_DIR / 'sweep_summary.csv'

for required_path in [
    TRAIN_CSV, TEST_CSV, TRAIN_SS3_PATH, TEST_SS3_PATH,
    POSITIVES_SPLITB_CSV, BASELINE_CHECKPOINT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required file: {required_path}')

SS3_HIGH_CONFIG = RSARegularizationConfig(
    lambda_cls=1.0,
    lambda_rsa_values=(7.0, 10.0, 15.0, 20.0),
    add_special_tokens=False,
    feature_key='ss3_structured',
)

seed_everything(RANDOM_STATE)
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Feature: {SS3_HIGH_CONFIG.feature_key}  (1.0=H/E structured, 0.0=C coil)')
print(f'λ values: {SS3_HIGH_CONFIG.lambda_rsa_values}')
print(f'Output dir: {SS3_HIGH_RESULTS_DIR}')

Device: cuda
Feature: ss3_structured  (1.0=H/E structured, 0.0=C coil)
λ values: (7.0, 10.0, 15.0, 20.0)
Output dir: /content/drive/MyDrive/XAllergen2.0/results/ss3_high_lambda


---
## Section A — Data Loading

In [4]:
train_df = pd.read_csv(TRAIN_CSV).copy()
test_df  = pd.read_csv(TEST_CSV).copy()
for df in [train_df, test_df]:
    df['sequence_id'] = df['sequence_id'].astype(str).str.strip()
    df['sequence']    = df['sequence'].astype(str).str.strip().str.upper()
    df['label']       = df['label'].astype(int)

display(inspect_rsa_inputs(
    train_rsa_path=TRAIN_SS3_PATH,
    test_rsa_path=TEST_SS3_PATH,
    train_frame=train_df,
    test_frame=test_df,
))

,path,format,compressed,n_sequences,rsa_min,rsa_max,rsa_in_unit_interval,min_length,max_length,expected_sequences,missing_sequences,extra_sequences,exact_id_match
0,/content/XAllergen2.0/data/ss3/deepalgpro_trai...,precomputed_rsa_json,True,5551,0.0,1.0,True,11,999,5551,0,0,True
1,/content/XAllergen2.0/data/ss3/deepalgpro_test...,precomputed_rsa_json,True,1377,0.0,1.0,True,11,979,1377,0,0,True


In [5]:
seed_everything(RANDOM_STATE)
train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=train_df['label'],
)
train_split_df = train_split_df.reset_index(drop=True).copy()
val_split_df   = val_split_df.reset_index(drop=True).copy()

print(f'Train: {len(train_split_df):,} | Val: {len(val_split_df):,} | Test: {len(test_df):,}')

Train: 4,995 | Val: 556 | Test: 1,377


In [6]:
train_lookup, test_lookup, lookup_summary = load_rsa_lookup_dicts(
    train_rsa_path=TRAIN_SS3_PATH,
    test_rsa_path=TEST_SS3_PATH,
    train_frame=train_df,
    test_frame=test_df,
    add_special_tokens=SS3_HIGH_CONFIG.add_special_tokens,
)
display(pd.DataFrame([lookup_summary['train'], lookup_summary['test']]))

,path,format,compressed,n_expected_sequences,n_parsed_sequences,n_missing_sequences,n_extra_sequences,n_length_mismatches,missing_sequence_ids,extra_sequence_ids,length_mismatches,add_special_tokens
0,/content/XAllergen2.0/data/ss3/deepalgpro_trai...,precomputed_rsa_json,True,5551,5551,0,0,0,[],[],[],False
1,/content/XAllergen2.0/data/ss3/deepalgpro_test...,precomputed_rsa_json,True,1377,1377,0,0,0,[],[],[],False


---
## Section B — High-λ Sweep

λ values: **7, 10, 15, 20**.  
Each run starts from the frozen ESM-2 baseline checkpoint and trains with the SS3-structured
penalty on the same train/val split used in notebook 13.

In [7]:
sweep_df = run_lambda_rsa_sweep(
    config=SS3_HIGH_CONFIG,
    train_split_df=train_split_df,
    val_split_df=val_split_df,
    test_df=test_df,
    train_rsa_lookup=train_lookup,
    test_rsa_lookup=test_lookup,
    positives_splitb_csv=POSITIVES_SPLITB_CSV,
    output_dir=SS3_HIGH_RESULTS_DIR,
    device=DEVICE,
).loc[:, [
    'lambda_rsa', 'epoch',
    'val_auroc', 'val_precision', 'val_recall', 'val_f1', 'val_mcc', 'val_accuracy',
    'test_auroc', 'test_precision', 'test_recall', 'test_f1', 'test_mcc', 'test_accuracy',
    'residue_auroc', 'residue_auprc', 'residue_precision_at_k',
]].copy()
sweep_df.to_csv(SS3_HIGH_SWEEP_CSV, index=False)
display(sweep_df)
print(f'Saved to: {SS3_HIGH_SWEEP_CSV}')
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


lambda_rsa=7:   0%|          | 0/30 [00:00<?, ?epoch/s]

Epoch   1/30 | train_cls=0.43613 | train_rsa=0.00381 | train_total=0.46282 | val_cls=0.34420 | val_rsa=0.00333 | val_total=0.36748 | val_auroc=0.94131 | best=1
Epoch   2/30 | train_cls=0.31449 | train_rsa=0.00382 | train_total=0.34122 | val_cls=0.28250 | val_rsa=0.00332 | val_total=0.30572 | val_auroc=0.95723 | best=2
Epoch   3/30 | train_cls=0.27684 | train_rsa=0.00377 | train_total=0.30326 | val_cls=0.27679 | val_rsa=0.00326 | val_total=0.29961 | val_auroc=0.95993 | best=3
Epoch   4/30 | train_cls=0.25637 | train_rsa=0.00372 | train_total=0.28240 | val_cls=0.25138 | val_rsa=0.00319 | val_total=0.27373 | val_auroc=0.96487 | best=4
Epoch   5/30 | train_cls=0.23919 | train_rsa=0.00365 | train_total=0.26471 | val_cls=0.25252 | val_rsa=0.00317 | val_total=0.27469 | val_auroc=0.96389 | best=4
Epoch   6/30 | train_cls=0.22773 | train_rsa=0.00366 | train_total=0.25337 | val_cls=0.23397 | val_rsa=0.00314 | val_total=0.25598 | val_auroc=0.96872 | best=6
Epoch   7/30 | train_cls=0.21914 | train

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

...

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluating proteins:   0%|          | 0/61 [00:00<?, ?it/s]

Evaluating proteins: processed 5/61 proteins
Evaluating proteins: processed 10/61 proteins
Evaluating proteins: processed 15/61 proteins
Evaluating proteins: processed 20/61 proteins
Evaluating proteins: processed 25/61 proteins
Evaluating proteins: processed 30/61 proteins
Evaluating proteins: processed 35/61 proteins
Evaluating proteins: processed 40/61 proteins
Evaluating proteins: processed 45/61 proteins
Evaluating proteins: processed 50/61 proteins
Evaluating proteins: processed 55/61 proteins
Evaluating proteins: processed 60/61 proteins
Evaluating proteins: processed 61/61 proteins


,lambda_rsa,epoch,val_auroc,val_precision,val_recall,val_f1,val_mcc,val_accuracy,test_auroc,test_precision,test_recall,test_f1,test_mcc,test_accuracy,residue_auroc,residue_auprc,residue_precision_at_k
0,7.0,24,0.982876,0.934545,0.941392,0.937956,0.877697,0.938849,0.967705,0.902616,0.924107,0.913235,0.828834,0.914306,0.5400,0.2909,0.2693
1,10.0,24,0.980533,0.930403,0.930403,0.930403,0.863265,0.931655,0.964647,0.897547,0.925595,0.911355,0.824667,0.912128,0.5521,0.2988,0.2759
2,15.0,19,0.978475,0.933579,0.926740,0.930147,0.863270,0.931655,0.964273,0.887624,0.928571,0.907636,0.816461,0.907771,0.5551,0.3002,0.2766
3,20.0,11,0.976339,0.926471,0.923077,0.924771,0.852466,0.926259,0.961552,0.881844,0.910714,0.896047,0.794199,0.896877,0.5541,0.3005,0.2839


Saved to: /content/drive/MyDrive/XAllergen2.0/results/ss3_high_lambda/sweep_summary.csv


In [8]:
best = sweep_df.loc[sweep_df['val_mcc'].idxmax()]
print('=== SS3-structured (high λ): best λ by val MCC ===')
print(f'  Best λ       : {best["lambda_rsa"]}')
print(f'  Val MCC      : {best["val_mcc"]:.4f}')
print(f'  Test MCC     : {best["test_mcc"]:.4f}')
print(f'  Test AUROC   : {best["test_auroc"]:.4f}')
print(f'  Residue AUROC: {best["residue_auroc"]:.4f}')

=== SS3-structured (high λ): best λ by val MCC ===
  Best λ       : 7.0
  Val MCC      : 0.8777
  Test MCC     : 0.8288
  Test AUROC   : 0.9677
  Residue AUROC: 0.5400


---
## Section C — Comparison Table

Rows (all selected by best val MCC within each sweep):
- **Baseline (λ=0)** — unregularized model from the RSA sweep (notebook 12)
- **SS3 λ ∈ {0.1–5}** — notebook 13 sweep
- **SS3 λ ∈ {7–20}** — this notebook

In [ ]:
RSA_SWEEP_CSV = RESULTS_DIR / 'rsa_regularization' / 'sweep_summary.csv'
SS3_SWEEP_CSV = RESULTS_DIR / 'ss3_regularization' / 'sweep_summary.csv'

for p in [RSA_SWEEP_CSV, SS3_SWEEP_CSV]:
    if not p.exists():
        raise FileNotFoundError(
            f'{p} not found. Run notebooks 12 and 13 first and ensure their output CSVs are present.'
        )


def _row(csv_path, label, select_best=True):
    df  = pd.read_csv(csv_path)
    row = df.loc[df['val_mcc'].idxmax()] if select_best else df.loc[df['lambda_rsa'].eq(0.0)].iloc[0]
    return {
        'Constraint':           label,
        'Best λ':               row['lambda_rsa'],
        'Val MCC':              round(float(row['val_mcc']),              4),
        'Test MCC':             round(float(row['test_mcc']),             4),
        'Test AUROC':           round(float(row['test_auroc']),           4),
        'Residue AUROC':        round(float(row['residue_auroc']),        4),
        'Residue AUPRC':        round(float(row['residue_auprc']),        4),
        'Residue Prec@k':       round(float(row['residue_precision_at_k']), 4),
    }


def _row_from_df(df, label):
    row = df.loc[df['val_mcc'].idxmax()]
    return {
        'Constraint':           label,
        'Best λ':               row['lambda_rsa'],
        'Val MCC':              round(float(row['val_mcc']),              4),
        'Test MCC':             round(float(row['test_mcc']),             4),
        'Test AUROC':           round(float(row['test_auroc']),           4),
        'Residue AUROC':        round(float(row['residue_auroc']),        4),
        'Residue AUPRC':        round(float(row['residue_auprc']),        4),
        'Residue Prec@k':       round(float(row['residue_precision_at_k']), 4),
    }


comparison_df = pd.DataFrame([
    _row(RSA_SWEEP_CSV,  'Baseline (λ=0)',       select_best=False),
    _row(SS3_SWEEP_CSV,  'SS3 λ∈{0.1–5}',        select_best=True),
    _row_from_df(sweep_df, 'SS3 λ∈{7–20}'),
]).set_index('Constraint')

print('=== SS3-structured: extended λ comparison ===')
display(comparison_df)

=== SS3-structured: extended λ comparison ===


,Best λ,Val MCC,Test MCC,Test AUROC,Residue AUROC,Residue AUPRC,Residue Prec@k
Constraint,,,,,,,
Baseline (λ=0),0.0,0.8454,0.8336,0.9699,0.4677,0.2529,0.2163
SS3 λ∈{0.1–5},5.0,0.8777,0.8446,0.9684,0.5300,0.2863,0.2649
SS3 λ∈{7–20},7.0,0.8777,0.8288,0.9677,0.5400,0.2909,0.2693


: 

In [ ]:
# Shut down the Colab runtime to free GPU after training completes.
import os
os.kill(os.getpid(), 9)

: 

: 